# CROCO hindcast analysis — circulation, upwelling and coastal processes
### Gulf of Guinea `GoG_12` (1/12°) and its nested child `IGOG_36` (1/36°), 1–15 June 2026

Built on the GGOSSS-2025 CROCO training notebook (`Python-Tutorial/croco_training/croco_train.ipynb`): the same tools
(xarray, xgcm for the C-grid, cartopy), the same first quantities (surface fields, wind stress, an Ekman upwelling
index), extended to the two hindcast configurations of this course.

| part | what | main variables |
|---|---|---|
| 1 | the two configurations: grid, bathymetry, period | `h`, `mask_rho`, `pm`, `pn` |
| 2 | circulation: surface currents, a vertical section, vorticity | `u`, `v`, `temp` |
| 3 | upwelling: wind stress, coastal Ekman transport, SST and thermocline response, vertical velocity | `sustr`, `svstr`, `temp`, `w` |
| 4 | coastal processes: river plumes, tides, surface boundary layer | `salt`, `zeta`, `hbl` |
| 5 | 1/12° vs 1/36° over the common area | all |

**Inputs** — the hindcast history files `croco_his.nc` (hourly records with tides):
`hindcast/model-runs/<CONFIG>/<YYYYMMDD>_<build>/hcast/CROCO_FILES/croco_his.nc`.
**Time**: CROCO stores `time` in seconds since **Yorig = 1993-01-01** (the files' units just say `second`).

## 0. Set-up
Run the notebook in the `ggosss26` environment. Change `BUILD` if you ran another physics combination
(`plain`, `plain_tides`, `plain_rivers`), and `PERIOD` to analyse another window. GoG_12 was run one extra day
(to 17 June) only to feed its child: the analysis keeps 1–15 June for both.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import xarray as xr
import xgcm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
from scipy.ndimage import distance_transform_edt, gaussian_filter
import warnings
warnings.filterwarnings("ignore", message="invalid value encountered")   # NaN over land and near the equator (f = 0)
warnings.filterwarnings("ignore", category=UserWarning, message="No artists with labels")

YORIG  = 1993                                   # time origin of every hindcast file
BUILD  = "plain_tides_rivers"                   # the binary the hindcast was run with
PERIOD = ("2026-06-01T00:00", "2026-06-15T23:59")
RHO0   = 1025.0                                 # reference density (kg m-3)
ROOT   = os.path.expanduser("~/croco_work/hindcast/model-runs")
FIGS   = os.path.expanduser("~/croco_work/notebooks/figures/analysis"); os.makedirs(FIGS, exist_ok=True)

FILES = {name: sorted(glob.glob(f"{ROOT}/{name}/2026????_{BUILD}/hcast/CROCO_FILES/croco_his.nc"))
         for name in ["GoG_12", "IGOG_36"]}
for k, v in FILES.items():
    print(k, "->", v if v else "NO FILE FOUND (run the hindcast first)")

### Helper functions
* `open_croco` opens the history file(s) lazily (dask chunks of 24 hourly records: a 1/36° 3-D field is several GB),
  converts `time` to dates with Yorig, removes duplicated records at cycle joins and keeps `PERIOD`.
* `make_grid` declares CROCO's **Arakawa C-grid** to xgcm: `u` sits between rho points in x (`xi_u`), `v` in y
  (`eta_v`). `to_rho` averages them back onto the rho points (as in the reference notebook, with explicit axes).
* `to_east_north` rotates grid-oriented vectors (along ξ, η) to east/north with the grid `angle`.
* `z_rho` computes the depth of every sigma level (CROCO `Vtransform = 2`):
  $z = \zeta + (\zeta + h)\,\dfrac{h_c\,s + h\,C_s(s)}{h_c + h}$.
* `depth_of` finds, in each column, the depth at which a field crosses a value (e.g. the 20 °C isotherm), and
  `at_depth` interpolates a field to a fixed depth.

In [ ]:
def open_croco(files, yorig=YORIG, period=PERIOD, chunks={"time": 24}):
    ds = xr.open_mfdataset(files, combine="nested", concat_dim="time", decode_times=False, chunks=chunks,
                           data_vars="minimal", coords="minimal", compat="override")
    t = pd.Timestamp(f"{yorig}-01-01") + pd.to_timedelta(ds["time"].values, unit="s")
    ds = ds.assign_coords(time=("time", t))
    _, keep = np.unique(ds["time"].values, return_index=True)
    ds = ds.isel(time=keep)
    if period is not None:
        ds = ds.sel(time=slice(*period))
    return ds.set_coords([c for c in ["lon_rho", "lat_rho", "lon_u", "lat_u", "lon_v", "lat_v"] if c in ds])

def make_grid(ds):
    return xgcm.Grid(ds, coords={"X": {"center": "xi_rho", "inner": "xi_u"},
                                 "Y": {"center": "eta_rho", "inner": "eta_v"}},
                     padding="extend", autoparse_metadata=False)   # xgcm >= 0.9: padding replaces periodic/boundary

def to_rho(grid, da):
    if "xi_u" in da.dims:
        return grid.interp(da, "X")
    if "eta_v" in da.dims:
        return grid.interp(da, "Y")
    return da

def to_east_north(ds, ax, ay):
    a = ds["angle"]
    return ax * np.cos(a) - ay * np.sin(a), ax * np.sin(a) + ay * np.cos(a)

def z_rho(ds, zeta):
    cs = ds["Cs_r"] if "Cs_r" in ds else ds["Cs_rho"]
    hc, h = float(ds["hc"]), ds["h"]
    z0 = (hc * ds["s_rho"] + h * cs) / (hc + h)
    return (zeta + (zeta + h) * z0).transpose("s_rho", ...)        # (s, eta, xi) or (s, eta) on a section

def depth_of(field, z, value):
    '''Depth (m, positive) where a (s, eta, xi) field first crosses `value` going down from the surface.'''
    F, Z = np.asarray(field)[::-1], np.asarray(z)[::-1]          # top -> bottom
    below = F < value
    k = np.argmax(below, axis=0)
    ok = below.any(axis=0) & (k > 0) & (F[0] >= value)
    k = np.clip(k, 1, None)
    f0, f1 = np.take_along_axis(F, (k - 1)[None], 0)[0], np.take_along_axis(F, k[None], 0)[0]
    z0, z1 = np.take_along_axis(Z, (k - 1)[None], 0)[0], np.take_along_axis(Z, k[None], 0)[0]
    zc = z0 + (value - f0) * (z1 - z0) / np.where(f1 != f0, f1 - f0, np.nan)
    return np.where(ok, -zc, np.nan)

def at_depth(field, z, depth):
    '''Linear interpolation of a (s, eta, xi) field to z = -depth (NaN below the sea floor).'''
    F, Z = np.asarray(field), np.asarray(z)
    zt = -abs(depth)
    k = np.clip(np.argmax(Z >= zt, axis=0), 1, Z.shape[0] - 1)   # first level at or above the target
    z0, z1 = np.take_along_axis(Z, (k - 1)[None], 0)[0], np.take_along_axis(Z, k[None], 0)[0]
    f0, f1 = np.take_along_axis(F, (k - 1)[None], 0)[0], np.take_along_axis(F, k[None], 0)[0]
    out = f0 + (zt - z0) * (f1 - f0) / np.where(z1 != z0, z1 - z0, np.nan)
    return np.where(Z[0] <= zt, out, np.nan)

def land(ax, ds):
    '''Grey land from the model mask + cartopy coastline and gridlines.'''
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.6)
    gl = ax.gridlines(draw_labels=True, lw=0.2, color="k"); gl.top_labels = gl.right_labels = False
    ax.pcolormesh(ds.lon_rho, ds.lat_rho, np.where(ds.mask_rho == 0, 1, np.nan), cmap="Greys", vmin=0, vmax=2,
                  transform=ccrs.PlateCarree(), shading="auto")

PC = ccrs.PlateCarree()

### Open both configurations
Opening is lazy: nothing large is read until a figure needs it. The printout shows the number of records, their
spacing (1 h with tides) and the grid size.

In [ ]:
DS, GRID = {}, {}
for name, files in FILES.items():
    if not files:
        continue
    ds = open_croco(files)
    DS[name], GRID[name] = ds, make_grid(ds)
    dt = pd.Series(ds.time.values).diff().median()
    dx = 1e-3 / float(ds.pm.where(ds.mask_rho == 1).mean())
    print(f"{name:8s} {ds.sizes['xi_rho']}x{ds.sizes['eta_rho']}x{ds.sizes['s_rho']}  mean dx {dx:.2f} km  "
          f"{ds.sizes['time']} records every {dt}  {str(ds.time.values[0])[:16]} -> {str(ds.time.values[-1])[:16]}")

---
## 1. The two configurations
The bathymetry `h` and the land mask are in every history file. The black frame on the GoG_12 map is the IGOG_36
domain: the child sits inside the parent, at least 0.3° from its edges, and takes its open-boundary values from it.

In [ ]:
fig, axes = plt.subplots(1, len(DS), figsize=(7 * len(DS), 5.5), subplot_kw={"projection": PC}, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (name, ds) in zip(axes, DS.items()):
    pc = ax.pcolormesh(ds.lon_rho, ds.lat_rho, ds.h.where(ds.mask_rho == 1), cmap=cmocean.cm.deep,
                       norm=mcolors.LogNorm(10, 6000), transform=PC, shading="auto")
    land(ax, ds); ax.set_title(f"{name}: bathymetry (m)")
    fig.colorbar(pc, ax=ax, shrink=0.8)
if "IGOG_36" in DS and "GoG_12" in DS:
    c = DS["IGOG_36"]
    lo, la = c.lon_rho.values, c.lat_rho.values
    axes[0].plot([lo.min(), lo.max(), lo.max(), lo.min(), lo.min()], [la.min(), la.min(), la.max(), la.max(), la.min()],
                 "k-", lw=1.5, transform=PC)
plt.savefig(f"{FIGS}/1_bathymetry.png", dpi=150); plt.show()

---
## 2. Circulation
### 2.1 Mean surface currents
The period mean over 15 days (≈ 29 M2 cycles) removes most of the tidal currents and keeps the sub-tidal
circulation: the eastward **Guinea Current** along the northern coast (≈ 3–5°N), the westward **South Equatorial
Current** around the equator, and the coastal flows of the Bight of Biafra.
`u`, `v` are moved to rho points (`to_rho`) and rotated to east/north before plotting.

In [ ]:
SURF = {}
for name, ds in DS.items():
    g = GRID[name]
    u = to_rho(g, ds.u.isel(s_rho=-1)).mean("time"); v = to_rho(g, ds.v.isel(s_rho=-1)).mean("time")
    ue, vn = to_east_north(ds, u, v)
    SURF[name] = dict(ue=ue.compute(), vn=vn.compute(), sst=ds.temp.isel(s_rho=-1).mean("time").compute(),
                      sss=ds.salt.isel(s_rho=-1).mean("time").compute())

fig, axes = plt.subplots(1, len(DS), figsize=(7 * len(DS), 5.5), subplot_kw={"projection": PC}, constrained_layout=True)
for ax, (name, ds) in zip(np.atleast_1d(axes), DS.items()):
    S = SURF[name]; spd = np.hypot(S["ue"], S["vn"]).where(ds.mask_rho == 1)
    pc = ax.pcolormesh(ds.lon_rho, ds.lat_rho, spd, cmap=cmocean.cm.speed, vmin=0, vmax=0.8, transform=PC, shading="auto")
    k = max(1, ds.sizes["xi_rho"] // 35)                           # ~35 arrows across
    ax.quiver(ds.lon_rho.values[::k, ::k], ds.lat_rho.values[::k, ::k], S["ue"].values[::k, ::k], S["vn"].values[::k, ::k],
              scale=8, width=0.0025, transform=PC)
    land(ax, ds); ax.set_title(f"{name}: mean surface current, {PERIOD[0][:10]} – {PERIOD[1][:10]}")
    fig.colorbar(pc, ax=ax, shrink=0.8, label="m s$^{-1}$")
plt.savefig(f"{FIGS}/2_1_surface_currents.png", dpi=150); plt.show()

### 2.2 Vertical section of zonal velocity and temperature at 6°E
A meridional section common to both grids. It crosses the equator, where the eastward **Equatorial Undercurrent**
flows beneath the westward surface current, centred near the top of the thermocline (≈ 50–100 m); the 20 °C
isotherm marks the thermocline. Depths come from `z_rho` with the mean sea level.

In [ ]:
LON_SECTION, ZMAX = 6.0, 300
fig, axes = plt.subplots(1, len(DS), figsize=(7 * len(DS), 4.5), constrained_layout=True, squeeze=False)
for ax, (name, ds) in zip(axes[0], DS.items()):
    i = int(np.argmin(np.abs(ds.lon_rho.mean("eta_rho").values - LON_SECTION)))
    g = GRID[name]
    u = to_rho(g, ds.u).isel(xi_rho=i).mean("time"); v = to_rho(g, ds.v).isel(xi_rho=i).mean("time")
    ue, _ = to_east_north(ds.isel(xi_rho=i), u, v)
    T = ds.temp.isel(xi_rho=i).mean("time")
    z = z_rho(ds.isel(xi_rho=i), ds.zeta.isel(xi_rho=i).mean("time")).compute()
    lat = np.broadcast_to(ds.lat_rho.isel(xi_rho=i).values, z.shape)
    wet = ds.mask_rho.isel(xi_rho=i).values == 1
    pc = ax.pcolormesh(lat, z, np.where(wet, ue.compute(), np.nan), cmap=cmocean.cm.balance, vmin=-0.6, vmax=0.6, shading="gouraud")
    cs = ax.contour(lat, z, np.where(wet, T.compute(), np.nan), levels=[14, 17, 20, 23, 26], colors="k", linewidths=0.7)
    ax.clabel(cs, fmt="%d°C", fontsize=7)
    ax.set_ylim(-ZMAX, 0); ax.set_xlabel("latitude"); ax.set_ylabel("z (m)")
    ax.set_title(f"{name}: zonal velocity (colour) and temperature, {float(ds.lon_rho.isel(xi_rho=i).mean()):.2f}°E")
    fig.colorbar(pc, ax=ax, label="u east (m s$^{-1}$)")
plt.savefig(f"{FIGS}/2_2_section_6E.png", dpi=150); plt.show()

### 2.3 Surface relative vorticity — what the finer grid resolves
$\zeta = \partial v/\partial x - \partial u/\partial y$, computed on rho points with the grid metrics
($\partial/\partial x = p_m\,\partial/\partial\xi$, $\partial/\partial y = p_n\,\partial/\partial\eta$), at one instant
of the last day (a daily mean would blur the eddies). Values are in units of $10^{-5}$ s$^{-1}$ (≈ f at 4°).

In [ ]:
def vorticity(ds, grid, t):
    u = to_rho(grid, ds.u.isel(s_rho=-1).sel(time=t, method="nearest")).values
    v = to_rho(grid, ds.v.isel(s_rho=-1).sel(time=t, method="nearest")).values
    dvdxi = np.gradient(v, axis=1); dudeta = np.gradient(u, axis=0)
    return np.where(ds.mask_rho == 1, ds.pm.values * dvdxi - ds.pn.values * dudeta, np.nan)

T_SNAP = pd.Timestamp(PERIOD[1]) - pd.Timedelta(hours=12)
fig, axes = plt.subplots(1, len(DS), figsize=(7 * len(DS), 5.5), subplot_kw={"projection": PC}, constrained_layout=True)
for ax, (name, ds) in zip(np.atleast_1d(axes), DS.items()):
    zeta_v = vorticity(ds, GRID[name], T_SNAP) / 1e-5
    pc = ax.pcolormesh(ds.lon_rho, ds.lat_rho, zeta_v, cmap=cmocean.cm.curl, vmin=-3, vmax=3, transform=PC, shading="auto")
    land(ax, ds); ax.set_title(f"{name}: surface vorticity, {T_SNAP:%Y-%m-%d %H:%M}")
    fig.colorbar(pc, ax=ax, shrink=0.8, label=r"$\zeta$ ($10^{-5}$ s$^{-1}$)")
plt.savefig(f"{FIGS}/2_3_vorticity.png", dpi=150); plt.show()

---
## 3. Upwelling
### 3.1 Wind stress
`sustr`, `svstr` (N m⁻²) are the surface stress CROCO computed from ERA5 with its bulk formulae, on u and v points,
along the grid axes: they are moved to rho points and rotated to east/north (as in the reference notebook, plus
the rotation). In June the south-easterly trades cross the equator and turn into the south-westerly monsoon flow.

In [ ]:
TAU = {}
for name, ds in DS.items():
    g = GRID[name]
    tx, ty = to_rho(g, ds.sustr), to_rho(g, ds.svstr)
    te, tn = to_east_north(ds, tx, ty)
    TAU[name] = dict(te=te, tn=tn)                                  # lazy (time, eta, xi)

fig, axes = plt.subplots(1, len(DS), figsize=(7 * len(DS), 5.5), subplot_kw={"projection": PC}, constrained_layout=True)
for ax, (name, ds) in zip(np.atleast_1d(axes), DS.items()):
    te, tn = TAU[name]["te"].mean("time").compute(), TAU[name]["tn"].mean("time").compute()
    mod = np.hypot(te, tn).where(ds.mask_rho == 1)
    pc = ax.pcolormesh(ds.lon_rho, ds.lat_rho, mod, cmap="cubehelix_r", vmin=0, vmax=0.08, transform=PC, shading="auto")
    k = max(1, ds.sizes["xi_rho"] // 30)
    ax.quiver(ds.lon_rho.values[::k, ::k], ds.lat_rho.values[::k, ::k], te.values[::k, ::k], tn.values[::k, ::k],
              scale=1.0, width=0.0025, transform=PC)
    land(ax, ds); ax.set_title(f"{name}: mean wind stress")
    fig.colorbar(pc, ax=ax, shrink=0.8, label="|τ| (N m$^{-2}$)")
plt.savefig(f"{FIGS}/3_1_wind_stress.png", dpi=150); plt.show()

### 3.2 Coastal upwelling index: offshore Ekman transport
The wind stress drives an Ekman transport per unit length (m² s⁻¹), to the right of the wind in the northern
hemisphere and to the left in the southern one:
$$M_x = \frac{\tau_y}{\rho_0 f},\qquad M_y = -\frac{\tau_x}{\rho_0 f}.$$
Where this transport points **offshore**, water is replaced from below: coastal upwelling. The index is the
offshore component $UI = \mathbf{M}\cdot\mathbf{n}$, with $\mathbf{n}$ the unit vector pointing from land to sea
(from the smoothed land mask), in a coastal band of about 25 km. $UI > 0$ = upwelling-favourable,
$UI < 0$ = downwelling-favourable.

* The reference notebook's $UI = \tau_y/(\rho_0 f)$ is the special case of a north–south coast ($\mathbf{n}$ = ±x).
  The Côte d'Ivoire–Ghana coast runs **east–west** (sea to the south): there the index is
  $-M_y = \tau_x/(\rho_0 f)$, upwelling for an eastward wind component.
* $f = 2\Omega\sin\varphi$ vanishes at the equator, where Ekman theory fails: cells within 2° of the equator are
  left out.

In [ ]:
def coast_band(ds, width_km=25.0):
    m = ds.mask_rho.values.astype(float)
    dx_km = 1e-3 / np.nanmean(np.where(m == 1, ds.pm.values, np.nan))
    band = (m == 1) & (distance_transform_edt(m) <= max(1.5, width_km / dx_km))
    ms = gaussian_filter(m, 2)
    gy, gx = np.gradient(ms)                                       # d/deta, d/dxi (index space)
    dmdx, dmdy = gx * ds.pm.values, gy * ds.pn.values              # per metre, along the grid axes
    ne, nn = to_east_north(ds, xr.DataArray(dmdx, dims=("eta_rho", "xi_rho")), xr.DataArray(dmdy, dims=("eta_rho", "xi_rho")))
    norm = np.hypot(ne, nn)
    return band, (ne / norm).values, (nn / norm).values           # n points from land to sea

UI = {}
for name, ds in DS.items():
    band, ne, nn = coast_band(ds)
    f = ds.f.values
    ok = band & (np.abs(ds.lat_rho.values) >= 2.0)
    Mx = TAU[name]["tn"] / (RHO0 * f); My = -TAU[name]["te"] / (RHO0 * f)
    UI[name] = ((Mx * ne + My * nn).where(ok)).resample(time="1D").mean().compute()   # daily, m2 s-1

fig, axes = plt.subplots(1, len(DS), figsize=(7 * len(DS), 5.5), subplot_kw={"projection": PC}, constrained_layout=True)
for ax, (name, ds) in zip(np.atleast_1d(axes), DS.items()):
    pc = ax.pcolormesh(ds.lon_rho, ds.lat_rho, UI[name].mean("time"), cmap=cmocean.cm.balance, vmin=-2, vmax=2,
                       transform=PC, shading="auto")
    land(ax, ds); ax.set_title(f"{name}: coastal upwelling index (offshore Ekman transport)")
    fig.colorbar(pc, ax=ax, shrink=0.8, label="UI (m$^2$ s$^{-1}$), > 0 upwelling")
plt.savefig(f"{FIGS}/3_2_upwelling_index.png", dpi=150); plt.show()

### 3.3 The ocean's answer: coastal SST, the thermocline and vertical velocity
Two coastal boxes, one per upwelling region of each configuration:
* **Ghana – Côte d'Ivoire** (GoG_12 only): 3.5°W–1.5°E, within 1° of the coast — the northern Gulf of Guinea
  upwelling, which starts in June–July;
* **Gabon – Congo** (both configurations): 8.5–12°E, 5°S–1°S.

For each box, the daily coastal SST is compared with the same box's offshore SST (2–3° farther out) and with the
upwelling index. Upwelling appears as a **coastal–offshore SST difference that grows negative** when UI > 0, a
**shallower 20 °C isotherm** at the coast, and **positive vertical velocity** a few tens of metres below the surface.

In [ ]:
BOXES = {   # name: (lon_min, lon_max, lat_min, lat_max) of the coastal box, offshore shift (dlon, dlat) in degrees
    "Ghana - Côte d'Ivoire": ((-3.5, 1.5, 3.5, 6.5), (0.0, -2.5)),
    "Gabon - Congo":         ((8.5, 12.0, -5.0, -1.0), (-2.5, 0.0)),
}
def box_mask(ds, b, band=None):
    lo, la = ds.lon_rho.values, ds.lat_rho.values
    m = (lo >= b[0]) & (lo <= b[1]) & (la >= b[2]) & (la <= b[3]) & (ds.mask_rho.values == 1)
    return m & band if band is not None else m

rows = []
fig, axes = plt.subplots(len(BOXES), 1, figsize=(11, 3.4 * len(BOXES)), constrained_layout=True, squeeze=False)
for ax, (bname, (b, (dlo, dla))) in zip(axes[:, 0], BOXES.items()):
    ax2 = ax.twinx()
    for name, ds in DS.items():
        band, _, _ = coast_band(ds, width_km=100)
        mc = box_mask(ds, b, band)
        mo = box_mask(ds, (b[0] + dlo, b[1] + dlo, b[2] + dla, b[3] + dla))
        if mc.sum() < 5 or mo.sum() < 5:
            continue                                                # box outside this domain
        sst = ds.temp.isel(s_rho=-1).resample(time="1D").mean()
        dsst = (sst.where(mc).mean(("eta_rho", "xi_rho")) - sst.where(mo).mean(("eta_rho", "xi_rho"))).compute()
        ui = UI[name].where(mc).mean(("eta_rho", "xi_rho"))
        ax.plot(dsst.time, dsst, "-o", ms=3, label=f"{name}: SST coast − offshore")
        ax2.plot(ui.time, ui, "--", label=f"{name}: UI")
        rows.append([bname, name, float(dsst.mean()), float(ui.mean()), float(np.corrcoef(dsst, ui)[0, 1]) if ui.notnull().all() else np.nan])
    ax.axhline(0, color="k", lw=0.5); ax.set_ylabel("ΔSST (°C)"); ax2.set_ylabel("UI (m$^2$ s$^{-1}$)")
    ax.set_title(bname); ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)
plt.savefig(f"{FIGS}/3_3_coastal_sst_ui.png", dpi=150); plt.show()
pd.DataFrame(rows, columns=["box", "config", "mean ΔSST (°C)", "mean UI (m2/s)", "corr(ΔSST, UI)"]).round(3)

Depth of the 20 °C isotherm (thermocline proxy, commonly used in the tropical Atlantic) and vertical velocity at
30 m, both averaged over the period. A shallow D20 along a coast and w > 0 in the coastal band are the
signatures of upwelling; Kelvin waves travelling along the equator and the coast also lift the thermocline, so a
shallow D20 is not always local wind-driven upwelling — compare with UI.

In [ ]:
fig, axes = plt.subplots(2, len(DS), figsize=(7 * len(DS), 10), subplot_kw={"projection": PC}, constrained_layout=True, squeeze=False)
for j, (name, ds) in enumerate(DS.items()):
    T = ds.temp.mean("time").compute(); zeta_m = ds.zeta.mean("time").compute()
    z = z_rho(ds, zeta_m).values
    d20 = np.where(ds.mask_rho == 1, depth_of(T.values, z, 20.0), np.nan)
    w30 = np.where(ds.mask_rho == 1, at_depth(ds.w.mean("time").compute().values, z, 30.0), np.nan) * 86400
    pc = axes[0, j].pcolormesh(ds.lon_rho, ds.lat_rho, d20, cmap="viridis_r", vmin=10, vmax=80, transform=PC, shading="auto")
    land(axes[0, j], ds); axes[0, j].set_title(f"{name}: depth of the 20 °C isotherm"); fig.colorbar(pc, ax=axes[0, j], shrink=0.8, label="m")
    pc = axes[1, j].pcolormesh(ds.lon_rho, ds.lat_rho, w30, cmap=cmocean.cm.balance, vmin=-5, vmax=5, transform=PC, shading="auto")
    land(axes[1, j], ds); axes[1, j].set_title(f"{name}: vertical velocity at 30 m"); fig.colorbar(pc, ax=axes[1, j], shrink=0.8, label="w (m day$^{-1}$), > 0 up")
plt.savefig(f"{FIGS}/3_3_d20_w30.png", dpi=150); plt.show()

---
## 4. Coastal processes
### 4.1 River plumes
With `RIVERS=1` the GloFAS discharge of the Niger, Sanaga, Ogooué, Volta … enters as point sources of fresh water
(27 °C, S = 0). The plumes are the low surface-salinity areas; the **34 isohaline** is a common plume boundary.
The plume area is summed over the grid cells ($1/(p_m p_n)$ each) with S < 34, per day.

In [ ]:
fig, axes = plt.subplots(1, len(DS), figsize=(7 * len(DS), 5.5), subplot_kw={"projection": PC}, constrained_layout=True)
for ax, (name, ds) in zip(np.atleast_1d(axes), DS.items()):
    sss = SURF[name]["sss"].where(ds.mask_rho == 1)
    pc = ax.pcolormesh(ds.lon_rho, ds.lat_rho, sss, cmap=cmocean.cm.haline, vmin=28, vmax=36, transform=PC, shading="auto")
    ax.contour(ds.lon_rho, ds.lat_rho, sss, levels=[34], colors="w", linewidths=1.0, transform=PC)
    land(ax, ds); ax.set_title(f"{name}: mean sea-surface salinity (white: S = 34)")
    fig.colorbar(pc, ax=ax, shrink=0.8, label="PSU", extend="min")
plt.savefig(f"{FIGS}/4_1_sss.png", dpi=150); plt.show()

plt.figure(figsize=(10, 3.5))
for name, ds in DS.items():
    area = (1.0 / (ds.pm * ds.pn)) * 1e-6                          # km2 per cell
    sss_d = ds.salt.isel(s_rho=-1).resample(time="1D").mean()
    plume = area.where((sss_d < 34) & (ds.mask_rho == 1)).sum(("eta_rho", "xi_rho")).compute()
    plt.plot(plume.time, plume, "-o", ms=3, label=name)
plt.ylabel("area with S < 34 (km$^2$)"); plt.title("river plume area (whole domain of each configuration)"); plt.legend(); plt.grid(lw=0.3)
plt.savefig(f"{FIGS}/4_1_plume_area.png", dpi=150); plt.show()

### 4.2 Tides at the coast
Hourly sea level at one coastal point common to both grids (Bight of Biafra, off the Niger–Cross estuaries), and a
least-squares harmonic fit of the four main constituents. Over 15 days, M2 and S2 are just separable (the Rayleigh
criterion asks for 14.8 days), as are K1 and O1 (13.7 days); N2 and K2 would need about a month and are left out.
Phases are relative to the first record (UTC), without nodal corrections: compare amplitudes, and phases between
the two configurations only.

In [ ]:
POINT = (8.0, 4.3)                                                 # lon, lat
PERIODS_H = {"M2": 12.4206012, "S2": 12.0, "K1": 23.9344696, "O1": 25.8193417}

def nearest_wet(ds, lon0, lat0):
    d = (ds.lon_rho - lon0) ** 2 + (ds.lat_rho - lat0) ** 2
    d = d.where(ds.mask_rho == 1)
    j, i = np.unravel_index(int(np.argmin(d.fillna(1e9).values)), d.shape)
    return j, i

rows = []
fig, ax = plt.subplots(figsize=(11, 3.5))
for name, ds in DS.items():
    j, i = nearest_wet(ds, *POINT)
    ssh = ds.zeta.isel(eta_rho=j, xi_rho=i).compute()
    t_h = (ssh.time - ssh.time[0]) / np.timedelta64(1, "h")
    dt_h = float(np.median(np.diff(t_h))); span_d = float(t_h[-1]) / 24
    ax.plot(ssh.time, ssh, lw=0.8, label=f"{name} ({float(ds.lon_rho[j, i]):.2f}°E, {float(ds.lat_rho[j, i]):.2f}°N)")
    if dt_h > 1.5 or span_d < 14.8:
        print(f"{name}: records every {dt_h:.1f} h over {span_d:.1f} d -- too sparse/short for a harmonic fit"); continue
    cols = [np.ones_like(t_h)]
    for p in PERIODS_H.values():
        w = 2 * np.pi / p; cols += [np.cos(w * t_h), np.sin(w * t_h)]
    coef, *_ = np.linalg.lstsq(np.column_stack(cols), ssh.values, rcond=None)
    for k, c in enumerate(PERIODS_H):
        a, b = coef[1 + 2 * k], coef[2 + 2 * k]
        rows.append([name, c, np.hypot(a, b), np.degrees(np.arctan2(b, a)) % 360])
ax.set_ylabel("sea level (m)"); ax.legend(fontsize=8); ax.grid(lw=0.3); ax.set_title("hourly sea level at the coastal point")
plt.savefig(f"{FIGS}/4_2_tide_point.png", dpi=150); plt.show()
pd.DataFrame(rows, columns=["config", "constituent", "amplitude (m)", "phase (deg, rel. to t0)"]).round(3)

### 4.3 Surface boundary layer
`hbl` is the depth of the surface boundary layer computed by the KPP mixing scheme: the layer stirred by wind and
surface cooling. Shallow values mark stable, freshened or upwelled coastal waters; deep values the wind-mixed
open ocean.

In [ ]:
fig, axes = plt.subplots(1, len(DS), figsize=(7 * len(DS), 5.5), subplot_kw={"projection": PC}, constrained_layout=True)
for ax, (name, ds) in zip(np.atleast_1d(axes), DS.items()):
    hbl = ds.hbl.mean("time").where(ds.mask_rho == 1).compute()
    pc = ax.pcolormesh(ds.lon_rho, ds.lat_rho, hbl, cmap=cmocean.cm.deep, vmin=0, vmax=60, transform=PC, shading="auto")
    land(ax, ds); ax.set_title(f"{name}: mean surface boundary-layer depth (KPP)")
    fig.colorbar(pc, ax=ax, shrink=0.8, label="m")
plt.savefig(f"{FIGS}/4_3_hbl.png", dpi=150); plt.show()

---
## 5. 1/12° vs 1/36° over the common area
Both configurations over the IGOG_36 domain: the child should reproduce the parent's large scales (it is driven
by it) and add **smaller, sharper structures**. The statistics below are computed on each model's own grid over the
same box: mean and spatial standard deviation of the mean SST, the RMS surface vorticity at the snapshot of 2.3,
the plume area and the mean coastal upwelling index.

In [ ]:
if "IGOG_36" in DS:
    c = DS["IGOG_36"]; BOX = [float(c.lon_rho.min()), float(c.lon_rho.max()), float(c.lat_rho.min()), float(c.lat_rho.max())]
    rows = []
    fig, axes = plt.subplots(1, len(DS), figsize=(7 * len(DS), 6), subplot_kw={"projection": PC}, constrained_layout=True)
    for ax, (name, ds) in zip(np.atleast_1d(axes), DS.items()):
        m = box_mask(ds, BOX)
        sst = SURF[name]["sst"].values; zv = vorticity(ds, GRID[name], T_SNAP)
        area = 1e-6 / (ds.pm.values * ds.pn.values)
        rows.append([name, np.nanmean(sst[m]), np.nanstd(sst[m]), np.sqrt(np.nanmean((zv[m] / 1e-5) ** 2)),
                     np.nansum(area[m & (SURF[name]["sss"].values < 34)]),
                     float(UI[name].mean("time").where(xr.DataArray(m, dims=("eta_rho", "xi_rho"))).mean())])
        pc = ax.pcolormesh(ds.lon_rho, ds.lat_rho, np.where(m, sst, np.nan), cmap=cmocean.cm.thermal, vmin=24, vmax=30, transform=PC, shading="auto")
        land(ax, ds); ax.set_extent(BOX, crs=PC); ax.set_title(f"{name}: mean SST over the IGOG_36 domain")
        fig.colorbar(pc, ax=ax, shrink=0.8, label="°C")
    plt.savefig(f"{FIGS}/5_sst_common_box.png", dpi=150); plt.show()
    display(pd.DataFrame(rows, columns=["config", "mean SST (°C)", "spatial std SST (°C)", "RMS ζ (1e-5 s-1)",
                                        "plume area S<34 (km2)", "mean UI (m2/s)"]).round(3))

---
### Reading the results — checklist
* **Currents**: eastward Guinea Current along the northern coast, westward flow near the equator, an eastward
  undercurrent under it at 6°E.
* **Upwelling**: where UI > 0 over several days, the coastal SST falls below the offshore SST, D20 shoals and w > 0.
  In early June the northern Gulf of Guinea upwelling is only starting; the Gabon–Congo coast responds to the
  south-easterly trades.
* **Coast**: plumes fresher than 34 off the large rivers, M2 the dominant tide (semi-diurnal regime), shallow
  boundary layers in the plumes.
* **Resolution**: similar means, larger RMS vorticity and sharper fronts at 1/36°.

Figures are saved in `~/croco_work/notebooks/figures/analysis/`.